<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module340/Lab5.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 5 — Preparing the H₂ Configuration Mix
**Quantum Optimization and Simulation — VQE Laboratory Series**

Turn the VQE control knob and observe bonding/antibonding populations.

**Suggested use:** 10–15 minute instructor demonstration followed by approximately one hour of independent work.

**Notebook style:** Most code is supplied. Complete the small items marked **YOUR TURN** and answer the reflection questions.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. When orbital labels are written in the order `q0, q1, ...`, this notebook explicitly notes the convention.

## Learning objectives
- Use the reduced two-qubit H₂ encoding.
- Prepare a superposition of bonding and antibonding configurations.
- Interpret the variational parameter as a mixing control.
- Verify that the circuit stays in the physical \(|01\rangle,|10\rangle\) subspace.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096

def run_counts(qc, shots=SHOTS, noise_model=None):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=SEED).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

## Reduced encoding

\[
|01\rangle = 	ext{bonding configuration},\qquad
|10\rangle = 	ext{antibonding configuration}.
\]

The other two computational states are outside this simplified two-configuration model.

## Part A — An excitation-preserving ansatz

In [ ]:
from qiskit.circuit.library import XXPlusYYGate

def h2_ansatz(theta):
    qc = QuantumCircuit(2)

    # Prepare |01> in Qiskit's |q1 q0> display convention.
    qc.x(0)

    # RXX(theta) RYY(theta) mixes |01> and |10>.
    # Phase-adjusted XX+YY gate: real Givens-style mixing.
    qc.append(XXPlusYYGate(2*theta, beta=np.pi/2), [0, 1])

    return qc

theta = 0.35
qc = h2_ansatz(theta)

display(qc.draw("mpl"))
print(Statevector.from_instruction(qc))

## Part B — Sample the configuration probabilities

In [ ]:
angles = np.linspace(0, np.pi/2, 9)

p01 = []
p10 = []

for theta in angles:
    qc = h2_ansatz(theta)
    qc.measure_all()
    counts = run_counts(qc, shots=4096)
    total = sum(counts.values())

    p01.append(counts.get("01", 0)/total)
    p10.append(counts.get("10", 0)/total)

plt.plot(angles, p01, "o-", label="P(01): bonding")
plt.plot(angles, p10, "s-", label="P(10): antibonding")
plt.xlabel("theta")
plt.ylabel("Probability")
plt.legend()
plt.show()

**Expected:** the two probabilities exchange smoothly as \(	heta\) changes, while `00` and `11` remain essentially absent.

### YOUR TURN
Find a value of `theta` that produces approximately 80% bonding and 20% antibonding.

In [ ]:
theta_guess = 0.0  # TODO
qc = h2_ansatz(theta_guess)
qc.measure_all()
counts = run_counts(qc, shots=8192)
print(counts)

## Reflection
Is \(	heta\) itself a probability? Explain.

<details>
<summary><b>Instructor solution / suggested answer</b></summary>


    No. \(	heta\) is a circuit parameter/rotation angle. It controls amplitudes, and probabilities are obtained by squaring amplitude magnitudes. Depending on the gate convention, the probabilities follow trigonometric functions of \(	heta\).

    A value near \(	hetapprox 0.46\) radians gives roughly an 80/20 split for this specific `XXPlusYYGate(2*theta, beta=pi/2)` convention.

</details>